### What was removed from this notebook

- Duplicate `evaluate_lof_scores` (unused here, already superseded by
  the shared `evaluate_scores` helper in `03_training.ipynb`).
- An Isolation Forest reload + prediction cell that duplicated what
  `03_training.ipynb` already produces (`test_predictions_df` /
  `test_eval_df`).
- An IF "final ranking" cell that referenced `test_eval_df["if_score"]`,
  a column that never existed (the actual column is `anomaly_score`) --
  this cell could not run as written.
- A final LOF test-evaluation cell referencing `test_if_df` and
  `feature_columns_unsupervised`, neither of which is defined anywhere
  in the project -- also could not run as written.

The correct, working IF and LOF temporal test evaluations already live
at the end of `03_training.ipynb`. This notebook is now just the
Random Forest evaluation, kept separate since RF is trained/loaded
independently.

In [124]:
# ============================================================
# SHARED RF EVAL HELPER
# One function for both the random-split and temporal-split RF
# evaluations instead of copy-pasting the same metric block twice.
# ============================================================

def evaluate_rf_model(model, X_test, y_test, label):
    probs = model.predict_proba(X_test)[:, 1]
    preds = model.predict(X_test)

    roc = roc_auc_score(y_test, probs)
    pr = average_precision_score(y_test, probs)
    recall = recall_score(y_test, preds)
    f1 = f1_score(y_test, preds)

    print("=" * 60)
    print(label)
    print("=" * 60)
    print(f"Accuracy : {accuracy_score(y_test, preds):.4f}")
    print(f"Precision: {precision_score(y_test, preds):.4f}")
    print(f"Recall   : {recall_score(y_test, preds):.4f}")
    print(f"F1-score : {f1_score(y_test, preds):.4f}")
    print(f"ROC-AUC  : {roc:.4f}")
    print(f"PR-AUC   : {pr:.4f}\n")

    print("Confusion matrix:")
    print(confusion_matrix(y_test, preds))

    print("\nClassification report:")
    print(classification_report(y_test, preds, digits=4))

    return roc, pr, recall, f1


In [125]:
print("⟳ Loading Random Forest model (random split)...")
best_rf = joblib.load("models/random_forest/random_forest_best_randomsplit.joblib")
print("✅ Done.\n")

random_rf_roc_auc, random_rf_pr_auc, random_rf_recall, random_rf_f1 = evaluate_rf_model(
    best_rf,
    X_test_rf,
    y_test_rf,
    "Random Forest — Random split (Experiment 1)",
)


⟳ Loading Random Forest model (random split)...
✅ Done.

Random Forest — Random split (Experiment 1)
Accuracy : 0.9902
Precision: 0.9812
Recall   : 0.9175
F1-score : 0.9483
ROC-AUC  : 0.9972
PR-AUC   : 0.9842

Confusion matrix:
[[8388   16]
 [  75  834]]

Classification report:
              precision    recall  f1-score   support

           0     0.9911    0.9981    0.9946      8404
           1     0.9812    0.9175    0.9483       909

    accuracy                         0.9902      9313
   macro avg     0.9862    0.9578    0.9714      9313
weighted avg     0.9902    0.9902    0.9901      9313



In [126]:
print("⟳ Loading Random Forest model (temporal split, Experiment 2B)...")
exp2b_model = joblib.load("models/random_forest/random_forest_exp2b.joblib")
print("✅ Done.\n")

temporal_rf_roc, temporal_rf_pr, temporal_rf_recall, temporal_rf_f1 = evaluate_rf_model(
    exp2b_model,
    X_test_temporal,
    y_test_temporal,
    "Random Forest — Temporal split (Experiment 2B)",
)


⟳ Loading Random Forest model (temporal split, Experiment 2B)...
✅ Done.

Random Forest — Temporal split (Experiment 2B)
Accuracy : 0.9744
Precision: 0.9375
Recall   : 0.4779
F1-score : 0.6331
ROC-AUC  : 0.9595
PR-AUC   : 0.7047

Confusion matrix:
[[8420   13]
 [ 213  195]]

Classification report:
              precision    recall  f1-score   support

           0     0.9753    0.9985    0.9868      8433
           1     0.9375    0.4779    0.6331       408

    accuracy                         0.9744      8841
   macro avg     0.9564    0.7382    0.8099      8841
weighted avg     0.9736    0.9744    0.9704      8841



In [127]:
print("Feature importance (random-split RF):")
rf_search = joblib.load("models/random_forest/rf_random_search_randomsplit.joblib")
feature_importance = (
    pd.DataFrame({
        "feature": X_train_rf.columns,
        "importance": best_rf.named_steps["model"].feature_importances_
    }).sort_values("importance", ascending=False)
)
print(f"{feature_importance.head(20)}\n")

personal_values = feature_importance.loc[
    feature_importance["feature"].str[1:].astype(int) < 95,
    "importance"
].tolist()
print(
    f"Aggregated personal features' importance value: {sum(personal_values):.4f}\n"
    f"Aggregated neighborhood graph features' importance value: {1 - sum(personal_values):.4f}\n"
)

print("Saving the tuning results...")
cv_results = pd.DataFrame(rf_search.cv_results_)
cv_results = cv_results.sort_values("rank_test_score")
cv_results.to_csv(
    "models/random_forest/rf_random_search_results_randomsplit.csv",
    index=False,
)


Feature importance (random-split RF):
    feature  importance
52      f54    0.114984
4        f6    0.098554
13      f15    0.087939
54      f56    0.071851
89      f91    0.040306
162    f164    0.037538
40      f42    0.030103
64      f66    0.024218
58      f60    0.024190
131    f133    0.023910
159    f161    0.019922
15      f17    0.019390
60      f62    0.018203
46      f48    0.017777
17      f19    0.017490
66      f68    0.016675
42      f44    0.015183
48      f50    0.014429
79      f81    0.014227
65      f67    0.012103

Aggregated personal features' importance value: 0.7817
Aggregated neighborhood graph features' importance value: 0.2183

Saving the tuning results...


In [140]:
results = pd.DataFrame([
    {
        "model": "LOF",
        "experiment": "Temporal",
        "roc_auc": temporal_lof_roc,
        "pr_auc": temporal_lof_pr
    },
    {
        "model": "Isolation Forest",
        "experiment": "Temporal",
        "roc_auc": temporal_if_roc,
        "pr_auc": temporal_if_pr
    },
    {
        "model": "Random Forest",
        "experiment": "Temporal",
        "roc_auc": temporal_rf_roc,   
        "pr_auc": temporal_rf_pr,
        "recall": temporal_rf_recall,
        "f1": temporal_rf_f1
    },
    {
        "model": "LOF",
        "experiment": "Random label holdout",
        "roc_auc": random_lof_roc_auc,
        "pr_auc": random_lof_pr_auc
    },
    {
        "model": "Isolation Forest",
        "experiment": "Random label holdout",
        "roc_auc": random_if_roc_auc,
        "pr_auc": random_if_pr_auc
    },
    {
        "model": "Random Forest",
        "experiment": "Random label holdout",
        "roc_auc": random_rf_roc_auc,  
        "pr_auc": random_rf_pr_auc,
        "recall": random_rf_recall,
        "f1": random_rf_f1
    }
])

display(results)

results.to_csv(
    "models/models_comparison.csv",
    index=False
)

print("Saved.")


,model,experiment,roc_auc,pr_auc,recall,f1
0,LOF,Temporal,0.460650,0.047858,NaN,NaN
1,Isolation Forest,Temporal,0.280879,0.029305,NaN,NaN
2,Random Forest,Temporal,0.959476,0.704713,0.477941,0.633117
3,LOF,Random label holdout,0.227117,0.058514,NaN,NaN
4,Isolation Forest,Random label holdout,0.094723,0.052174,NaN,NaN
5,Random Forest,Random label holdout,0.997159,0.984200,0.917492,0.948266


Saved.
